# V7.1 — Test YOLO26m reviewed trên val và test cố định

Notebook tải V7.1 best từ Camber, đánh giá cả val/test và so trực tiếp với seed85 best trên cùng dataset. Kết quả per-class được lưu CSV/JSON và upload lại Camber.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, json, time
import pandas as pd

EXPERIMENT = 'finding_v7p1_171126'
RUN_NAME = 'yolo26m_finetune_v2_reviewed_181126'
REMOTE_RUN = f'stash://bunpmc/projects/{EXPERIMENT}/{RUN_NAME}'
BASELINE_REMOTE = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'
IMGSZ, BATCH = 640, 8

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir(): return p
    raise RuntimeError('Mở notebook từ repo train')

ROOT = find_repo()
os.chdir(ROOT)
DATA_ROOT = ROOT / 'benchmakring_transmission_line_defect_model/data/finding_nemo_human_reviewed_6cls_v2_171126'
DATA_YAML = DATA_ROOT / 'data_local.yaml'
assert (DATA_ROOT / 'REVIEW_APPLIED_181126.json').is_file(), 'Đây chưa phải dataset đã áp dụng review 18/11'
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / 'images' / split
    label_dir = DATA_ROOT / 'labels' / split
    assert image_dir.is_dir() and label_dir.is_dir(), f'Dataset thiếu split {split}'
    image_stems = {p.stem for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}
    label_stems = {p.stem for p in label_dir.glob('*.txt') if p.is_file()}
    assert image_stems, f'Không có ảnh trong {image_dir}'
    missing_labels = image_stems - label_stems
    assert not missing_labels, f'{split}: {len(missing_labels)} ảnh thiếu label, ví dụ: {sorted(missing_labels)[:5]}'
    print(f'{split}: {len(image_stems)} images, {len(label_stems)} labels')
DATA_YAML.write_text(
    'path: ' + DATA_ROOT.resolve().as_posix() + '\n'
    'train: images/train\nval: images/val\ntest: images/test\n'
    'names:\n  0: insulator\n  1: insulator_broken\n  2: insulator_flashover\n'
    '  3: bird_nest\n  4: foreign_object\n  5: broken_strand\n',
    encoding='utf-8')
OUT = ROOT / 'benchmakring_transmission_line_defect_model/results/finding_v7p1_171126' / RUN_NAME
WEIGHTS = ROOT / 'weights/v7p1'
OUT.mkdir(parents=True, exist_ok=True); WEIGHTS.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.is_file(), DATA_YAML
print('Dataset:', DATA_YAML)
print('Output :', OUT)

In [ ]:
# %pip install "ultralytics==8.4.172" pandas python-dotenv
import torch, ultralytics
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT/'.env')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Không thấy CUDA'
print('Ultralytics:',ultralytics.__version__)
print('GPU:',torch.cuda.get_device_name(0))

def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'): return
    for line in (ROOT/'.env').read_text(encoding='utf-8').splitlines():
        key, sep, value = line.partition('=')
        if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
            os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('Thiếu CAMBER_API_KEY trong .env')

def wsl_path(path: Path):
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

CAMBER = '/home/dotru/.camber/bin/camber'
load_camber_key()
CAMBER_ENV = os.environ.copy()
wslenv = [x for x in CAMBER_ENV.get('WSLENV','').split(':') if x]
if 'CAMBER_API_KEY' not in wslenv: wslenv.append('CAMBER_API_KEY')
CAMBER_ENV['WSLENV'] = ':'.join(wslenv)

def camber(*parts,check=True):
    converted = [wsl_path(x) if isinstance(x,Path) else str(x) for x in parts]
    p = subprocess.run(['wsl','--',CAMBER,'stash',*converted],env=CAMBER_ENV,capture_output=True, text=True, encoding='utf-8', errors='replace')
    if check and p.returncode: raise RuntimeError((p.stdout or p.stderr)[-3000:])
    return p

auth = subprocess.run(['wsl','--',CAMBER,'me'],env=CAMBER_ENV,capture_output=True, text=True, encoding='utf-8', errors='replace')
assert auth.returncode == 0,(auth.stdout or auth.stderr)[-1200:]
print('Camber authentication OK via WSL')


## Tải hai checkpoint

In [ ]:
MODELS = {
    'seed85_before': (BASELINE_REMOTE, ROOT / 'weights/v7p1_seed85_curated_yolo26m_best.pt'),
    'v7p1_after': (REMOTE_RUN + '/weights/best.pt', ROOT / 'runs/finding_v7p1_171126/yolo26m_finetune_v2_reviewed_181126/weights/best.pt'),
}
for name, (remote, local) in MODELS.items():
    if not local.is_file():
        print('Downloading', name, '...')
        camber('cp', remote, local)
    print(name, round(local.stat().st_size / 2**20, 1), 'MiB')

## Chạy val + test và lưu per-class CSV

In [ ]:
summary_rows, class_rows = [], []
for model_name, (_, checkpoint) in MODELS.items():
    model = YOLO(str(checkpoint))
    for split in ('val', 'test'):
        print(f'\n===== {model_name} / {split} =====')
        metrics = model.val(
            data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH, device=0,
            project=str(OUT / 'runs'), name=f'{model_name}_{split}', exist_ok=True,
            plots=True, save_json=False, verbose=True,
        )
        summary_rows.append({
            'model': model_name, 'split': split, 'precision': float(metrics.box.mp),
            'recall': float(metrics.box.mr), 'mAP50': float(metrics.box.map50),
            'mAP50_95': float(metrics.box.map),
        })
        names = metrics.names
        for i, cls in enumerate(metrics.box.ap_class_index.astype(int).tolist()):
            class_rows.append({
                'model': model_name, 'split': split, 'class_id': cls, 'class': names[cls],
                'precision': float(metrics.box.p[i]), 'recall': float(metrics.box.r[i]),
                'AP50': float(metrics.box.ap50[i]), 'AP50_95': float(metrics.box.ap[i]),
            })

summary = pd.DataFrame(summary_rows)
per_class = pd.DataFrame(class_rows)
summary.to_csv(OUT / 'benchmark_summary.csv', index=False)
per_class.to_csv(OUT / 'per_class_summary.csv', index=False)
(OUT / 'benchmark_results.json').write_text(json.dumps({
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': str(DATA_YAML), 'summary': summary_rows, 'per_class': class_rows,
}, indent=2), encoding='utf-8')
display(summary)
display(per_class)

## Chênh lệch V7.1 so với seed85 và upload kết quả

In [ ]:
before = summary[summary.model == 'seed85_before'].set_index('split')
after = summary[summary.model == 'v7p1_after'].set_index('split')
delta = (after[['precision','recall','mAP50','mAP50_95']] - before[['precision','recall','mAP50','mAP50_95']]).reset_index()
delta.insert(0, 'comparison', 'v7p1_after-minus-seed85_before')
delta.to_csv(OUT / 'delta_vs_seed85.csv', index=False)
display(delta)

REMOTE_EVAL = REMOTE_RUN + '/evaluation'
if camber('ls', REMOTE_EVAL, check=False).returncode != 0:
    camber('mkdir', REMOTE_EVAL, check=False)
for filename in ('benchmark_summary.csv', 'per_class_summary.csv', 'delta_vs_seed85.csv', 'benchmark_results.json'):
    camber('cp', OUT / filename, REMOTE_EVAL + '/' + filename)
print('Uploaded:', REMOTE_EVAL)